# Figure 2: Compressed Sensing Reconstruction

This section reproduces the intuitive reconstruction demonstration from
Lustig, Donoho & Pauly (2007), Figure 2.

The key insight is that **random** undersampling of k-space spreads aliasing
incoherently (noise-like), unlike uniform undersampling which creates coherent
ghost artefacts at predictable locations. This incoherence makes it possible
to separate genuine signal components from artefacts using simple thresholding.

The interactive figure below lets you vary:

- **Sampling pattern** — random, uniform, or both overlaid for comparison
- **Acceleration factor R** — how many times fewer k-space samples are acquired
- **Threshold σ** — how many standard deviations above the mean defines a
  "detected" component in each iteration

In [1]:
import numpy as np
from scipy.signal import find_peaks

# ── Step 1: Define a sparse signal ───────────────────────────────────────────
# A signal is "sparse" if most of its values are zero.
# Here, only 3 of 128 samples are non-zero.
N         = 128
positions = [10, 50, 100]    # locations of non-zero components
heights   = [1.0, 0.85, 0.40]

signal = np.zeros(N)
for p, h in zip(positions, heights):
    signal[p] = h

# ── Step 2: Full k-space ──────────────────────────────────────────────────────
# The DFT of a sparse signal is NOT sparse — energy spreads across all
# frequencies.  This is the k-space an MRI scanner would measure at full
# resolution.
kspace_full = np.fft.fft(signal)

# ── Step 3: Random undersampling (acceleration factor R) ─────────────────────
# Acquire only N/R k-space lines, chosen at random.
# Random sampling distributes aliasing incoherently (noise-like), unlike
# uniform spacing which creates coherent ghost artefacts at N/R intervals.
R   = 4                                            # 4× faster acquisition
rng = np.random.default_rng(42)
idx = np.sort(rng.choice(N, N // R, replace=False))
mask = np.zeros(N, dtype=bool)
mask[idx] = True

k_sampled = np.where(mask, kspace_full, 0)
x_recon   = np.fft.ifft(k_sampled).real   # amplitudes ≈ true / R

# ── Step 4: Iteration 1 — detect strong components ───────────────────────────
# Threshold at mean + 5σ: only genuine peaks stand above the incoherent noise.
sigma1 = 5.0
thr1   = x_recon.mean() + sigma1 * x_recon.std()

peaks1, _ = find_peaks(
    np.abs(np.where(np.abs(x_recon) >= thr1, x_recon, 0)),
    height=thr1 * 0.9,
)
x_rec1         = np.zeros(N)
x_rec1[peaks1] = x_recon[peaks1] * R     # rescale: zero-fill IFFT divides by R

# Subtract the detected component's k-space contribution from the data
k_residual = k_sampled - np.where(mask, np.fft.fft(x_rec1), 0)
x_residual = np.fft.ifft(k_residual).real

# ── Step 5: Iteration 2 — recover weaker components from residual ────────────
# The residual k-space no longer contains the strong components, so the
# threshold can be lowered to find weaker peaks.
sigma2 = 3.0
thr2   = x_residual.mean() + sigma2 * x_residual.std()

peaks2, _ = find_peaks(
    np.abs(np.where(np.abs(x_residual) >= thr2, x_residual, 0)),
    height=thr2 * 0.9,
)
x_rec2         = np.zeros(N)
x_rec2[peaks2] = x_residual[peaks2] * R

# ── Final reconstruction ──────────────────────────────────────────────────────
x_final = x_rec1.copy()
for p in peaks2:
    x_final[p] += x_rec2[p]

print(f"Iter 1 peaks: {sorted(peaks1.tolist())}")
print(f"Iter 2 peaks: {sorted(peaks2.tolist())}")
print(f"True positions: {positions}")

Iter 1 peaks: [10]
Iter 2 peaks: [50, 100]
True positions: [10, 50, 100]


In [2]:
import sys, pathlib
from IPython.display import HTML

_root = pathlib.Path.cwd()
if not (_root / 'figures').exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from figures.fig_02_interactive_v2 import make_embeddable_html

html = make_embeddable_html(r_values=[2, 3, 4], sigma_values=[2.0, 3.0, 4.0])
HTML(html)

**How to read the figure:**

- **Row 1** — True sparse signal (time domain) and its full k-space with
  the acquired sample locations highlighted.
- **Row 2** — Zero-filled IFFT reconstruction. The red dotted lines are the
  detection threshold. Peaks above the threshold are accepted as genuine
  signal components.
- **Row 3** — K-space of only the detected components.
- **Row 4** — Residual after subtracting the detected components' k-space
  contribution. A second (lower) threshold is applied.
- **Row 5** — Final reconstruction combining both iterations, with the
  ground-truth positions shown as open circles.

Try increasing **R** to see how stronger undersampling raises the noise floor
and makes recovery harder. Compare **Random** vs **Uniform** to see coherent
ghost artefacts appear in the uniform case.